# Hyperparameter Tuning — Home Credit

Tune regression models from `model_comparison.ipynb` to improve **R²** and reduce **MAE** / **RMSE** when predicting **AMT_CREDIT** (loan amount).

Models tuned: Ridge, Lasso, ElasticNet, Decision Tree, Random Forest. Linear Regression is kept as an untuned baseline.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
CV_FOLDS = 3

## 2. Load & prepare data

Same pipeline as `model_comparison.ipynb`: load `application_train.csv`, select target and features, `dropna()`, 80/20 split with `random_state=42`.

In [2]:
DATA_PATH = r"C:\Users\hchin\OneDrive\Desktop\ml_project\Machine Learning End to End Project\csv\application_train.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

target = "AMT_CREDIT"
feature_cols = [
    "AMT_INCOME_TOTAL",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CNT_CHILDREN",
    "DAYS_BIRTH",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

model_df = df[[target] + feature_cols].copy()
rows_before = len(model_df)
model_df = model_df.dropna()
rows_after = len(model_df)

print(f"Rows before dropna: {rows_before:,}")
print(f"Rows after dropna:  {rows_after:,}")
print(f"Dropped rows:       {rows_before - rows_after:,}")

X = model_df[feature_cols]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train size: {len(X_train):,} samples")
print(f"Test size:  {len(X_test):,} samples")

Dataset shape: (307511, 123)
Rows before dropna: 307,511
Rows after dropna:  109,483
Dropped rows:       198,028
Train size: 87,586 samples
Test size:  21,897 samples


## 3. Hyperparameter tuning

GridSearchCV (cv=3) for linear models; RandomizedSearchCV for Decision Tree (n_iter=15) and Random Forest (n_iter=10) to keep runtime manageable. Ridge, Lasso, and ElasticNet use a `StandardScaler` pipeline.

In [4]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return r2, mae, rmse


def run_search(name, search, X_train, y_train):
    print(f"Tuning {name}...")
    search.fit(X_train, y_train)
    print(f"  Best params: {search.best_params_}")
    print(f"  Best CV score (neg MSE): {search.best_score_:.2f}")
    print()
    return search.best_estimator_, search.best_params_


tuned_models = {}
best_params = {}

# Linear Regression — untuned baseline
lr = LinearRegression()
lr.fit(X_train, y_train)
tuned_models["Linear Regression"] = lr
best_params["Linear Regression"] = "default (no tuning)"
print("Linear Regression: fitted with default params (baseline)\n")

# Ridge
ridge_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    param_grid={"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
tuned_models["Ridge"], best_params["Ridge"] = run_search(
    "Ridge", ridge_search, X_train, y_train
)

# Lasso
lasso_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10000))]),
    param_grid={"model__alpha": [0.001, 0.01, 0.1, 1, 10]},
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
tuned_models["Lasso"], best_params["Lasso"] = run_search(
    "Lasso", lasso_search, X_train, y_train
)

# ElasticNet
en_search = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(max_iter=10000)),
    ]),
    param_grid={
        "model__alpha": [0.001, 0.01, 0.1, 1, 10],
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
    },
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
tuned_models["Elastic Net"], best_params["Elastic Net"] = run_search(
    "Elastic Net", en_search, X_train, y_train
)

# Decision Tree
dt_search = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid={
        "max_depth": [5, 10, 15, 20, None],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 5, 10],
    },
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
tuned_models["Decision Tree"], best_params["Decision Tree"] = run_search(
    "Decision Tree", dt_search, X_train, y_train
)

# Random Forest — RandomizedSearchCV for speed
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions={
        "n_estimators": [50, 100, 150, 200, 300],
        "max_depth": [5, 10, 15, 20, 25, None],
        "min_samples_split": [2, 5, 10, 20],
    },
    n_iter=10,
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
tuned_models["Random Forest"], best_params["Random Forest"] = run_search(
    "Random Forest", rf_search, X_train, y_train
)

Linear Regression: fitted with default params (baseline)

Tuning Ridge...
  Best params: {'model__alpha': 100}
  Best CV score (neg MSE): -4339277978.21

Tuning Lasso...
  Best params: {'model__alpha': 1}
  Best CV score (neg MSE): -4340175122.22

Tuning Elastic Net...
  Best params: {'model__alpha': 0.01, 'model__l1_ratio': 0.9}
  Best CV score (neg MSE): -4339343138.93

Tuning Decision Tree...
  Best params: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 20}
  Best CV score (neg MSE): -1736081375.48

Tuning Random Forest...
  Best params: {'n_estimators': 150, 'min_samples_split': 5, 'max_depth': None}
  Best CV score (neg MSE): -1286521995.61



## 4. Evaluate tuned models on test set

Compute **R²**, **MAE**, and **RMSE** for each model on the same held-out test set.

In [5]:
tuned_results = []
for name, model in tuned_models.items():
    r2, mae, rmse = evaluate_model(model, X_test, y_test)
    tuned_results.append({
        "Model": name,
        "Best Params": str(best_params[name]),
        "R²": r2,
        "MAE": mae,
        "RMSE": rmse,
    })
    print(f"{name}:")
    print(f"  Best params: {best_params[name]}")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAE:  {mae:,.2f}")
    print(f"  RMSE: {rmse:,.2f}")
    print()

tuned_df = pd.DataFrame(tuned_results).sort_values("R²", ascending=False).reset_index(drop=True)
tuned_df

Linear Regression:
  Best params: default (no tuning)
  R²:   0.9745
  MAE:  50,491.57
  RMSE: 65,922.50

Ridge:
  Best params: {'model__alpha': 100}
  R²:   0.9745
  MAE:  50,564.39
  RMSE: 65,923.58

Lasso:
  Best params: {'model__alpha': 1}
  R²:   0.9745
  MAE:  50,491.84
  RMSE: 65,922.48

Elastic Net:
  Best params: {'model__alpha': 0.01, 'model__l1_ratio': 0.9}
  R²:   0.9745
  MAE:  50,555.20
  RMSE: 65,923.01

Decision Tree:
  Best params: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 20}
  R²:   0.9910
  MAE:  17,697.06
  RMSE: 39,235.12

Random Forest:
  Best params: {'n_estimators': 150, 'min_samples_split': 5, 'max_depth': None}
  R²:   0.9936
  MAE:  16,369.70
  RMSE: 33,152.86



,Model,Best Params,R²,MAE,RMSE
0,Random Forest,"{'n_estimators': 150, 'min_samples_split': 5, ...",0.993555,16369.702722,33152.864335
1,Decision Tree,"{'max_depth': 20, 'min_samples_leaf': 2, 'min_...",0.990973,17697.057753,39235.122056
2,Lasso,{'model__alpha': 1},0.974516,50491.844514,65922.482907
3,Linear Regression,default (no tuning),0.974516,50491.569910,65922.500818
4,Elastic Net,"{'model__alpha': 0.01, 'model__l1_ratio': 0.9}",0.974516,50555.195264,65923.011896
5,Ridge,{'model__alpha': 100},0.974515,50564.388187,65923.580375


## 5. Before vs After comparison

Default metrics from `model_comparison.ipynb` compared with tuned results. Linear Regression is unchanged (baseline).

In [6]:
default_metrics = {
    "Linear Regression": {"R²": 0.9744, "MAE": 50775.59, "RMSE": 66058.89},
    "Ridge": {"R²": 0.9745, "MAE": 50491.62, "RMSE": 65922.50},
    "Lasso": {"R²": 0.9745, "MAE": 50491.94, "RMSE": 65922.48},
    "Elastic Net": {"R²": 0.9744, "MAE": 50761.98, "RMSE": 66049.44},
    "Decision Tree": {"R²": 0.9857, "MAE": 30245.74, "RMSE": 49318.78},
    "Random Forest": {"R²": 0.9872, "MAE": 29503.59, "RMSE": 46650.86},
}

comparison_rows = []
for _, row in tuned_df.iterrows():
    name = row["Model"]
    defaults = default_metrics[name]
    comparison_rows.append({
        "Model": name,
        "Best Params": row["Best Params"],
        "Default R²": defaults["R²"],
        "Tuned R²": row["R²"],
        "Δ R²": row["R²"] - defaults["R²"],
        "Default MAE": defaults["MAE"],
        "Tuned MAE": row["MAE"],
        "Δ MAE": row["MAE"] - defaults["MAE"],
        "Default RMSE": defaults["RMSE"],
        "Tuned RMSE": row["RMSE"],
        "Δ RMSE": row["RMSE"] - defaults["RMSE"],
    })

before_after_df = pd.DataFrame(comparison_rows).sort_values("Tuned R²", ascending=False).reset_index(drop=True)

display_df = before_after_df.copy()
for col in ["Default R²", "Tuned R²", "Δ R²"]:
    display_df[col] = display_df[col].map(lambda x: f"{x:.4f}")
for col in ["Default MAE", "Tuned MAE", "Δ MAE", "Default RMSE", "Tuned RMSE", "Δ RMSE"]:
    display_df[col] = display_df[col].map(lambda x: f"{x:,.2f}")

print("Before vs After Hyperparameter Tuning (sorted by Tuned R² descending)")
print("=" * 80)
print(display_df.to_string(index=False))

before_after_df

Before vs After Hyperparameter Tuning (sorted by Tuned R² descending)
            Model                                                       Best Params Default R² Tuned R²   Δ R² Default MAE Tuned MAE      Δ MAE Default RMSE Tuned RMSE     Δ RMSE
    Random Forest  {'n_estimators': 150, 'min_samples_split': 5, 'max_depth': None}     0.9872   0.9936 0.0064   29,503.59 16,369.70 -13,133.89    46,650.86  33,152.86 -13,498.00
    Decision Tree {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 20}     0.9857   0.9910 0.0053   30,245.74 17,697.06 -12,548.68    49,318.78  39,235.12 -10,083.66
            Lasso                                               {'model__alpha': 1}     0.9745   0.9745 0.0000   50,491.94 50,491.84      -0.10    65,922.48  65,922.48       0.00
Linear Regression                                               default (no tuning)     0.9744   0.9745 0.0001   50,775.59 50,491.57    -284.02    66,058.89  65,922.50    -136.39
      Elastic Net                  

,Model,Best Params,Default R²,Tuned R²,Δ R²,Default MAE,Tuned MAE,Δ MAE,Default RMSE,Tuned RMSE,Δ RMSE
0,Random Forest,"{'n_estimators': 150, 'min_samples_split': 5, ...",0.9872,0.993555,0.006355,29503.59,16369.702722,-13133.887278,46650.86,33152.864335,-13497.995665
1,Decision Tree,"{'max_depth': 20, 'min_samples_leaf': 2, 'min_...",0.9857,0.990973,0.005273,30245.74,17697.057753,-12548.682247,49318.78,39235.122056,-10083.657944
2,Lasso,{'model__alpha': 1},0.9745,0.974516,0.000016,50491.94,50491.844514,-0.095486,65922.48,65922.482907,0.002907
3,Linear Regression,default (no tuning),0.9744,0.974516,0.000116,50775.59,50491.569910,-284.020090,66058.89,65922.500818,-136.389182
4,Elastic Net,"{'model__alpha': 0.01, 'model__l1_ratio': 0.9}",0.9744,0.974516,0.000116,50761.98,50555.195264,-206.784736,66049.44,65923.011896,-126.428104
5,Ridge,{'model__alpha': 100},0.9745,0.974515,0.000015,50491.62,50564.388187,72.768187,65922.50,65923.580375,1.080375


## 6. Conclusion

Review the before/after table above to see which models gained the most from tuning. Tree-based models (Decision Tree, Random Forest) typically benefit most from hyperparameter search; regularized linear models may show smaller gains because defaults were already near-optimal for this feature set.

The **best overall model** is the one with the highest **Tuned R²** and lowest **Tuned MAE/RMSE** in the comparison table.

In [7]:
best_row = before_after_df.iloc[0]
most_improved = before_after_df.loc[before_after_df["Δ R²"].idxmax()]

print(f"Best overall model: {best_row['Model']}")
print(f"  Tuned R²:   {best_row['Tuned R²']:.4f}")
print(f"  Tuned MAE:  {best_row['Tuned MAE']:,.2f}")
print(f"  Tuned RMSE: {best_row['Tuned RMSE']:,.2f}")
print(f"  Best params: {best_row['Best Params']}")
print()
print(f"Largest R² improvement: {most_improved['Model']} (Δ R² = {most_improved['Δ R²']:+.4f})")

Best overall model: Random Forest
  Tuned R²:   0.9936
  Tuned MAE:  16,369.70
  Tuned RMSE: 33,152.86
  Best params: {'n_estimators': 150, 'min_samples_split': 5, 'max_depth': None}

Largest R² improvement: Random Forest (Δ R² = +0.0064)
